# Gacher Doctor - Custom Crop Disease Image Classification Model Training

This Google Colab notebook will guide you through training a custom **MobileNetV3-Large** Image Classifier to detect plant leaf diseases. Once trained, the model is exported to **TensorFlow.js (WebGL Layers)** format to run directly in the user's web browser, ensuring fast and 100% free offline-capable leaf diagnostic operations.

## Datasets trained:
1. **PlantVillage Dataset:** Tomato (Early/Late Blight, Healthy), Potato (Early/Late Blight, Healthy).
2. **Rice Leaf Disease Image Dataset:** Rice Blast, Brown Spot, Tungro, Healthy.

---

### Step 1: Install Dependencies & Setup Kaggle API
Run this cell to set up the necessary Python packages and upload your `kaggle.json` credentials.

In [ ]:
# Install tensorflowjs converter
!pip install tensorflowjs torchvision torch albumentations opencv-python-headless

import os
from google.colab import files

# Upload kaggle.json downloaded from Kaggle Settings page
if not os.path.exists('/content/kaggle.json'):
    print("Please upload your kaggle.json file:")
    uploaded = files.upload()
    !mkdir -p ~/.kaggle
    !cp kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json
    print("Kaggle API successfully configured!")

### Step 2: Download Crop Datasets from Kaggle
We pull the PlantVillage dataset and the Rice Leaf Disease image dataset directly from Kaggle's high-speed servers.

In [ ]:
# Download PlantVillage dataset (Tomato & Potato)
!kaggle datasets download -d aradhye/plantvillage-dataset -p /content/dataset --unzip

# Download Rice Leaf Disease dataset
!kaggle datasets download -d vbookshelf/rice-leaf-diseases -p /content/dataset-rice --unzip

print("Datasets successfully downloaded and extracted!")

### Step 3: Organize Images & Data Preprocessing
We restructure the downloaded images into class folders and apply Albumentations for image resizing, normalization, and advanced data augmentations.

In [ ]:
import glob
import shutil

# Set up a clean data directory
data_dir = '/content/crops_data'
os.makedirs(data_dir, exist_ok=True)

# Map Kaggle folders to our 8 core target classes
class_mappings = {
    # Tomato
    '/content/dataset/PlantVillage/Tomato_Early_blight': 'tomato_early_blight',
    '/content/dataset/PlantVillage/Tomato_Late_blight': 'tomato_late_blight',
    '/content/dataset/PlantVillage/Tomato_healthy': 'tomato_healthy',
    # Potato
    '/content/dataset/PlantVillage/Potato___Early_blight': 'potato_early_blight',
    '/content/dataset/PlantVillage/Potato___Late_blight': 'potato_late_blight',
    '/content/dataset/PlantVillage/Potato___healthy': 'potato_healthy',
    # Rice (from rice leaf diseases dataset)
    '/content/dataset-rice/rice_leaf_diseases/Bacterial leaf blight': 'rice_leaf_blight',
    '/content/dataset-rice/rice_leaf_diseases/Brown spot': 'rice_brown_spot',
    '/content/dataset-rice/rice_leaf_diseases/Leaf smut': 'rice_leaf_smut'
}

for src, dest_name in class_mappings.items():
    dest_path = os.path.join(data_dir, dest_name)
    if os.path.exists(src):
        shutil.copytree(src, dest_path, dirs_exist_ok=True)
        print(f"Organized {dest_name} (images: {len(os.listdir(dest_path))})")

print("Data directories successfully restructured!")

### Step 4: Define PyTorch Dataset, Augmentations & MobileNetV3-Large Model
We define the PyTorch dataset pipeline, split into Train/Validation, configure the MobileNetV3 transfer learning architecture, and set up the AdamW optimizer.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from PIL import Image
from sklearn.model_selection import train_test_split

# Hyperparameters
BATCH_SIZE = 32
EPOCHS = 10
LEARNING_RATE = 0.001
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Image Augmentations
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class LeafDataset(Dataset):
    def __init__(self, file_paths, labels, transform=None):
        self.file_paths = file_paths
        self.labels = labels
        self.transform = transform
        
    def __len__(self):
        return len(self.file_paths)
        
    def __getitem__(self, idx):
        img_path = self.file_paths[idx]
        img = Image.open(img_path).convert('RGB')
        label = self.labels[idx]
        
        if self.transform:
            img = self.transform(img)
            
        return img, label

# Gather all image paths and labels
classes = sorted(os.listdir(data_dir))
file_paths = []
labels = []
for idx, cls in enumerate(classes):
    cls_dir = os.path.join(data_dir, cls)
    for img_file in os.listdir(cls_dir):
        file_paths.append(os.path.join(cls_dir, img_file))
        labels.append(idx)

# Split into Train/Val
X_train, X_val, y_train, y_val = train_test_split(file_paths, labels, test_size=0.2, random_state=42, stratify=labels)

train_dataset = LeafDataset(X_train, y_train, transform=train_transform)
val_dataset = LeafDataset(X_val, y_val, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# Load pre-trained MobileNetV3-Large
model = models.mobilenet_v3_large(pretrained=True)
# Replace classification head
num_features = model.classifier[3].in_features
model.classifier[3] = nn.Linear(num_features, len(classes))
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)

print(f"MobileNetV3 Loaded! Target classes: {len(classes)} ({classes})")

### Step 5: Run Model Training & Validation Loop
Execute the training cycle. The GPU will accelerate this process, achieving high accuracy in under 15 minutes.

In [ ]:
print("Starting Model Training...")
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
    epoch_loss = running_loss / len(train_loader.dataset)
    epoch_acc = correct / total
    
    # Validation Loop
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()
            
    val_epoch_loss = val_loss / len(val_loader.dataset)
    val_epoch_acc = val_correct / val_total
    
    print(f"Epoch {epoch+1}/{EPOCHS} - Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f} | Val Loss: {val_epoch_loss:.4f} Val Acc: {val_epoch_acc:.4f}")

torch.save(model.state_dict(), 'gacher_doctor_mobilenet.pth')
print("Training Complete! PyTorch Model Saved.")

### Step 6: Export Model to TensorFlow.js (Layers Format)
We convert the trained PyTorch weights into ONNX/Keras first, then compile them into a browser-executable WebGL `model.json` layers format.

In [ ]:
# Export PyTorch model to ONNX
dummy_input = torch.randn(1, 3, 224, 224).to(DEVICE)
torch.onnx.export(model.to(DEVICE), dummy_input, 'model.onnx', 
                  export_params=True, opset_version=11, 
                  do_constant_folding=True, 
                  input_names=['input'], output_names=['output'])

# Convert ONNX to TensorFlow/Keras and then to tfjs
!pip install onnx onnx2tf tensorflow-decision-forests
!onnx2tf -i model.onnx -o keras_model

# Convert saved Keras model to TensorFlow.js Layers
!tensorflowjs_converter --input_format=tf_saved_model --output_node_names='output' --saved_model_tags=serve /content/keras_model /content/tfjs_model

print("TensorFlow.js model successfully generated!")

# Zip the model files for download
!zip -r gacher_doctor_tfjs_model.zip /content/tfjs_model
files.download('gacher_doctor_tfjs_model.zip')

### Step 7: Done!
Extract the downloaded zip archive. Copy the contents of the `tfjs_model` directory (which contains `model.json` and a few `.bin` weight shard files) into your Next.js project directory at `frontend/public/model/` so they are publicly accessible.